# Week 9 — Human judgment and model certainty

**Research task:** Inspect one model recommendation while its displayed certainty changes from 0.55 to 0.75 to 0.95. The evidence and recommendation remain fixed. Record whether the certainty display changes your decision.

**Python introduced:** the first small student-edited function, parameters, local variables, `return` and an `if/elif/else` branch controlled by a human choice.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session09/session09_llms_at_work.ipynb)

Run the setup cell below before anything else. Colab uses OpenRouter. Local JupyterLab uses Ollama by default.

## Prepare the notebook environment

The next cell installs missing SDKs in Colab. Locally it checks that Jupyter is using the course environment and prints the exact repair command if necessary.

In [ ]:
SESSION = "session09"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib as setup_importlib
import importlib.util as setup_importlib_util
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session05":
    course_packages["plotly"] = "plotly>=6.3,<7"
    course_packages["umap"] = "umap-learn>=0.5.9,<1"
    course_packages["pandas"] = "pandas>=2.3,<3"
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath(
        setup_os.getenv("COURSE_COLAB_ROOT", "/content/GenAI_Soc2026")
    )
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The complete GenAI_Soc2026 repository could not be found. A notebook "
            "downloaded by itself is not enough for local work. Download or clone the "
            "repository, open a terminal in that folder, and run: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Python executable:", setup_sys.executable)
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")
else:
    import json as setup_json
    import ollama as setup_ollama

    setup_config = setup_json.loads(
        (COURSE_ROOT / "config" / "course_models.json").read_text()
    )
    setup_local_model = setup_config["local"]["model"]
    try:
        setup_models = setup_ollama.list().models
        setup_model_names = [
            getattr(item, "model", None) or getattr(item, "name", None)
            for item in setup_models
        ]
        print("Ollama server: reachable at localhost:11434")
        if setup_local_model in setup_model_names:
            print("Course local model: ready —", setup_local_model)
        else:
            print("Course local model: NOT INSTALLED —", setup_local_model)
            print("NEXT STEP: open a terminal and run: ollama pull " + setup_local_model)
    except Exception as setup_error:
        print("Ollama server: NOT REACHABLE")
        print("NEXT STEP: start the Ollama application, then run: ollama list")
        print("Diagnostic:", str(setup_error).splitlines()[0])


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

try:
    import ollama
    from openrouter import OpenRouter
except ModuleNotFoundError:
    ollama = None
    OpenRouter = None

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config" / "course_models.json").exists():
    raise FileNotFoundError("Run this file from inside the complete GenAI_Soc2026 folder.")

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]
USE_RECORDED = os.getenv("COURSE_USE_RECORDED", "0") == "1"

## Store the workplace decision and evidence

The consultant must advise on a limited internal pilot. The dictionary contains the only evidence available to the model. It does not contain calibration, subgroup results or information about the costs of different errors.

In [ ]:
ROUTE = "openrouter" if IN_COLAB else "ollama"  # local default
decision_context = (
    "A consulting team must advise a lender about whether to begin a three-month "
    "internal pilot of a mortgage-prediction model. Staff would review every model "
    "recommendation during the pilot."
)
model_evidence = {
    "held_out_cases": 200,
    "model_accuracy": 0.80,
    "always_predict_no_mortgage_accuracy": 0.60,
    "calibration_reported": False,
    "subgroup_performance_reported": False,
    "error_costs_reported": False,
}
output_schema = {
    "type": "object",
    "properties": {
        "recommendation": {
            "type": "string",
            "enum": ["begin_limited_pilot", "request_more_evidence", "do_not_use"],
        },
        "evidence_used": {"type": "array", "items": {"type": "string"}},
        "missing_information": {"type": "array", "items": {"type": "string"}},
        "explanation": {"type": "string"},
    },
    "required": ["recommendation", "evidence_used", "missing_information", "explanation"],
    "additionalProperties": False,
}
recorded_raw_output = json.dumps({
    "recommendation": "request_more_evidence",
    "evidence_used": [
        "The model achieved 0.80 accuracy on 200 held-out cases.",
        "The stated baseline accuracy is 0.60.",
    ],
    "missing_information": [
        "calibration",
        "subgroup performance",
        "the costs of different errors",
    ],
    "explanation": (
        "The model exceeds the stated baseline, but the supplied evidence does not "
        "show whether its errors are acceptable for the proposed pilot."
    ),
})

## Ask the model for one recommendation

This cell makes the only model call. It prints the raw JSON string before parsing it. The same parsed recommendation will appear beside all three certainty displays.

In [ ]:
evidence_text = json.dumps(model_evidence, indent=2)
prompt = f"""You are assisting a consultant with this workplace decision:

{decision_context}

Available evidence:
{evidence_text}

Use only the supplied evidence. Choose one of the three permitted recommendations.
Identify the evidence you used and information that remains missing. Return JSON."""
messages = [{"role": "user", "content": prompt}]

if USE_RECORDED:
    raw_output = recorded_raw_output
    model_used = "recorded-teaching-return"
elif ROUTE == "openrouter":
    if OpenRouter is None:
        raise ModuleNotFoundError("Run 'uv sync --frozen' before using OpenRouter.")
    api_key = os.getenv("OPENROUTER_API_KEY") or getpass("OpenRouter course key (hidden): ")
    response_format = {"type": "json_schema", "json_schema": {
        "name": "workplace_recommendation",
        "strict": True,
        "schema": output_schema,
    }}
    with OpenRouter(api_key=api_key) as client:
        response = client.chat.send(
            model=HOSTED_MODEL,
            messages=messages,
            temperature=0,
            response_format=response_format,
        )
    raw_output = response.choices[0].message.content
    model_used = HOSTED_MODEL
elif ROUTE == "ollama":
    if ollama is None:
        raise ModuleNotFoundError("Run 'uv sync --frozen' before using Ollama.")
    response = ollama.chat(think=False,
        model=LOCAL_MODEL,
        messages=messages,
        format=output_schema,
        options={"temperature": 0},
    )
    raw_output = response.message.content
    model_used = LOCAL_MODEL
else:
    raise ValueError("ROUTE must be 'openrouter' or 'ollama'")

print("Raw model return:\n", raw_output)
print("Raw return type:", type(raw_output))
model_advice = json.loads(raw_output)
print("Parsed recommendation:", model_advice["recommendation"])
print("Parsed return type:", type(model_advice))

## Store three displays of the same advice

Edit the human actions and reasons so that they reflect what you would do. The certainty values change, but the evidence and recommendation do not.

In [ ]:
certainty_levels = [0.55, 0.75, 0.95]
print("Displayed certainty levels:", certainty_levels)
print("Evidence remains fixed:", model_evidence)
print("Recommendation remains fixed:", model_advice["recommendation"])

# These are editable placeholders. Replace them with your own decisions and reasons.
human_actions = [
    "ask_for_more_information",
    "escalate_to_specialist",
    "accept_recommendation",
]
human_reasons = [
    "The recommendation names missing evidence, so I would ask what can be supplied.",
    "The displayed certainty is higher, but a specialist should still assess the missing tests.",
    "I noticed that the high displayed certainty made me more willing to accept the advice.",
]

## Define the first small function

The function receives a recommendation, displayed certainty, human action and human reason. Its branch translates the selected action into an organizational next step.

In [ ]:
def record_human_judgment(
    recommendation,
    stated_certainty,
    human_action,
    human_reason,
):
    """Return one record of a worker's response to displayed model advice."""
    if human_action == "accept_recommendation":
        next_step = "Proceed with the recommendation."
    elif human_action == "ask_for_more_information":
        next_step = "Request additional evidence before deciding."
    elif human_action == "escalate_to_specialist":
        next_step = "Send the evidence and advice to a specialist."
    elif human_action == "reject_recommendation":
        next_step = "Do not proceed with the recommendation."
    else:
        raise ValueError(
            "human_action must be accept_recommendation, ask_for_more_information, "
            "escalate_to_specialist or reject_recommendation"
        )

    judgment_record = {
        "recommendation": recommendation,
        "stated_certainty": stated_certainty,
        "human_action": human_action,
        "human_reason": human_reason,
        "next_step": next_step,
    }
    return judgment_record

## Record one decision at each certainty level

The loop takes the action and reason at the same list position as the current certainty. The function returns one judgment dictionary, which is appended to the results list.

In [ ]:
judgment_records = []
for position, stated_certainty in enumerate(certainty_levels):
    human_action = human_actions[position]
    human_reason = human_reasons[position]
    record = record_human_judgment(
        recommendation=model_advice["recommendation"],
        stated_certainty=stated_certainty,
        human_action=human_action,
        human_reason=human_reason,
    )
    judgment_records.append(record)
    print("\nDisplayed certainty:", stated_certainty)
    print("Human action:", record["human_action"])
    print("Next step:", record["next_step"])

## Compare the worker's decisions

This cell prints all three records and verifies that the evidence and recommendation remained unchanged. Any difference in the action therefore requires a human explanation.

In [ ]:
for record in judgment_records:
    print(
        record["stated_certainty"],
        record["human_action"],
        "—",
        record["human_reason"],
    )

evidence_changed = False
recommendation_changed = False
print("Did the evidence change?", evidence_changed)
print("Did the recommendation change?", recommendation_changed)
human_action_changed = not (
    human_actions[0] == human_actions[1] == human_actions[2]
)
print("Did the human action change?", human_action_changed)

## Reflect on judgment and organizational visibility

The questions connect reliance on the recommendation to Barley, Zuboff and Wiles et al.

In [ ]:
reflection_questions = [
    "Did higher displayed certainty change your willingness to follow the recommendation?",
    "Did the evidence justify that change?",
    "What validation would make the certainty value meaningful?",
    "What can the organization now observe about your decisions and reasons?",
    "What could you explain later without the assistant?",
]
for question in reflection_questions:
    print(question)

## A short unassisted check

Answer this without asking the model. A displayed certainty value has no frequency interpretation unless someone has validated how that number relates to outcomes.

In [ ]:
knowledge_check = (
    "Does a displayed certainty of 0.95 mean that this recommendation is correct 95% "
    "of the time? What evidence would be needed to interpret that number?"
)
print(knowledge_check)

# ONE CHANGE: edit one value in human_actions or human_reasons, rerun, and compare.

## Your one change and recording

Change one action or reason, rerun and explain the function inputs, branch and returned record. State whether certainty language changed your judgment despite unchanged evidence.